# 06 — Bubble wall tracking: 1+1D vs 3+1D

For each available time snapshot in both BubbleMaster (1+1D) and sledgehamr (3+1D),
finds the outer bubble wall position by locating where φ crosses the φ(R_out) threshold
from the instanton profile.  Compares wall trajectories to diagnose any systematic offset.

In [ ]:
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path

import ptbuilder as pt

REPO_ROOT = Path(pt.__file__).parent.parent

PYSLEDGEHAMR_PATH = '/global/cfs/cdirs/m3166/buschman/sledgehamr'
sys.path.append(PYSLEDGEHAMR_PATH)
import pySledgehamr as sledgehamr

config  = pt.Config()
sh_base = '/pscratch/sd/b/buschman/other_runs/'

# --- BubbleMaster runs -------------------------------------------------------
BM_RUNS = [
    dict(lambda_bar=0.84,  d=55.55),
    dict(lambda_bar=0.069, d=12.54),
]

# --- sledgehamr runs ---------------------------------------------------------
# Coordinate convention (from FirstOrderPhaseTransition.cpp):
#   B.x = zlocs[b] - zlocs[0]   → AMReX x = HDF5 zloc
#   B.y = ylocs[b] - ylocs[0]   → AMReX y = HDF5 yloc
#   B.z = xlocs[b] - xlocs[0]   → AMReX z = HDF5 xloc
#
# GetSlice(axis='x') returns the AMReX y-z plane at AMReX x = L/2, i.e.:
#   rows  = HDF5 yloc   → select via sh_y_center
#   cols  = HDF5 xloc   → collision axis, spans [0, L]
#   fixed = HDF5 zloc at L/2
#
# sh_z_center: actual HDF5 zloc of the bubbles.  For lb=0.84 this equals L/2
# exactly; for lb=0.069 it is 22.5 vs L/2=25, so the slice misses the bubble
# centre by dz=2.5.  The 3D outer-wall radius is recovered as
#   R = sqrt(raw_x_distance^2 + dz^2)
#
# x_left / x_right: HDF5 xloc of the left / right bubble whose outer wall is
# to be measured.  Set to None when that bubble is absent from the run.
#
# Bubble positions (from nb04):
#   lb=0.84 : b0=(72.25,89.92,100)  b1=(127.8,89.92,100)
#   lb=0.069: b0=(16.23,19.06,22.5) b1=(28.77,19.06,22.5)
SH_RUNS = [
    # lb=0.84 — all three runs carry all three bubbles
    dict(lambda_bar=0.84, run_id='N3_234_0_1',
         sh_path=sh_base+'bubble_N3_234_0_1_',
         sh_L=200., sh_axis='x', sh_lev=0,
         sh_y_center=89.92, sh_z_center=100.0,
         x_left=72.25, x_right=127.8),
    dict(lambda_bar=0.84, run_id='N3_234_0_2',
         sh_path=sh_base+'bubble_N3_234_0_2_',
         sh_L=200., sh_axis='x', sh_lev=0,
         sh_y_center=89.92, sh_z_center=100.0,
         x_left=72.25, x_right=127.8),
    dict(lambda_bar=0.84, run_id='N3_234_0_1_2',
         sh_path=sh_base+'bubble_N3_234_0_1_2',
         sh_L=200., sh_axis='x', sh_lev=0,
         sh_y_center=89.92, sh_z_center=100.0,
         x_left=72.25, x_right=127.8),
    # lb=0.069 — each run contains only the named pair
    dict(lambda_bar=0.069, run_id='N3_456_0_1',
         sh_path=sh_base+'bubble_N3_456_0_1_',
         sh_L=50., sh_axis='x', sh_lev=0,
         sh_y_center=19.06, sh_z_center=22.5,
         x_left=16.23, x_right=28.77),
    dict(lambda_bar=0.069, run_id='N3_456_1_2',
         sh_path=sh_base+'bubble_N3_456_1_2_',
         sh_L=50., sh_axis='x', sh_lev=0,
         sh_y_center=19.06, sh_z_center=22.5,
         x_left=None,   # b0 not in this run
         x_right=28.77),
    dict(lambda_bar=0.069, run_id='N3_456_0_2',
         sh_path=sh_base+'bubble_N3_456_0_2_',
         sh_L=50., sh_axis='x', sh_lev=0,
         sh_y_center=19.06, sh_z_center=22.5,
         x_left=16.23,
         x_right=None),  # b1 not in this run
]

LAMBDA_VALS = sorted({r['lambda_bar'] for r in BM_RUNS}, reverse=True)

lambda_info = {}
for r in BM_RUNS:
    lb      = r['lambda_bar']
    d       = r['d']
    model   = pt.PhysicsModel(pt.Phi4Potential(lb), config)
    t_coll  = float(model.kinematics.collision_time(d))
    gamma   = float(model.kinematics.Gamma(t_coll))
    profile = model.instanton
    lambda_info[lb] = dict(
        d=d, t_coll=t_coll, gamma=gamma,
        phi_threshold=float(profile.interp(profile.rout_0)),
        rout_0=profile.rout_0,
        bm_tag=f'lb{lb}_g{gamma:.4g}',
        model=model,
    )
    print(f'lb={lb}  d={d}  t_coll={t_coll:.3f}  '
          f'rout_0={profile.rout_0:.3f}  phi_thr={lambda_info[lb]["phi_threshold"]:.4f}')

## 1. BubbleMaster wall tracking

In [ ]:
def find_outer_wall(z, phi, threshold):
    """Return outermost z where phi > threshold; NaN if none."""
    idx = np.where(phi > threshold)[0]
    return float(z[idx[-1]]) if len(idx) > 0 else np.nan


bm_tracks = {}
for lb, info in lambda_info.items():
    tag   = info['bm_tag']
    model = info['model']
    out_dir = model.results_dir / 'validation' / tag / 'out'
    fields_path = out_dir / 'fields.h5'
    if not fields_path.exists():
        print(f'[{tag}]  fields.h5 not found — run notebook 04 first')
        continue
    with h5py.File(fields_path, 'r') as f:
        s_bm   = f['s'][:]
        z_bm   = f['z'][:]
        phi_bm = f['phi'][:]
    thr    = info['phi_threshold']
    d_half = info['d'] / 2.
    dz_bm  = float(z_bm[1] - z_bm[0])   # BM grid spacing → measurement precision
    walls = np.array([find_outer_wall(z_bm, phi_bm[i, :], thr) - d_half
                      for i in range(len(s_bm))])
    bm_tracks[lb] = dict(t=s_bm, r_wall=walls, dz=dz_bm)
    valid = np.isfinite(walls)
    print(f'lb={lb}  {valid.sum()}/{len(s_bm)} BM snapshots with wall  '
          f's=[{s_bm[0]:.3f},{s_bm[-1]:.3f}]  dz_bm={dz_bm:.4f}')

## 2. sledgehamr wall tracking

In [ ]:
sh_tracks = {}
for b in SH_RUNS:
    lb      = b['lambda_bar']
    info    = lambda_info[lb]
    thr     = info['phi_threshold']
    L       = b['sh_L']
    x_left  = b['x_left']    # None → bubble absent, skip left measurement
    x_right = b['x_right']   # None → bubble absent, skip right measurement
    y_c     = b['sh_y_center']
    dz      = abs(b['sh_z_center'] - L / 2.)

    output  = sledgehamr.Output(b['sh_path'])
    t_snaps = output.GetTimesOfSlices()

    t_out       = []
    r_left_out  = []
    r_right_out = []
    dx_run      = np.nan   # grid spacing; constant at lev=0, set on first valid snap
    n_fail      = 0

    for idx in range(len(t_snaps)):
        try:
            slices = output.GetSlice(idx, b['sh_axis'], b['sh_lev'], ['Phi'])
            t      = slices['t']
            phi_2d = slices['Phi']
            n_rows, n_cols = phi_2d.shape

            if np.isnan(dx_run):
                dx_run = L / n_cols   # grid spacing = measurement precision

            row_idx = int(np.clip(round(y_c / L * n_rows), 0, n_rows - 1))
            phi_1d  = phi_2d[row_idx, :]
            z_box   = np.linspace(0, L, n_cols)

            idx_above = np.where(phi_1d > thr)[0]

            # Left outer wall: raw = x-distance only; sqrt adds z-offset correction.
            if len(idx_above) == 0 or x_left is None:
                r_l = np.nan
            else:
                raw = x_left - z_box[idx_above[0]]
                r_l = np.sqrt(raw**2 + dz**2) if raw > 0 else np.nan

            # Right outer wall.
            if len(idx_above) == 0 or x_right is None:
                r_r = np.nan
            else:
                raw = z_box[idx_above[-1]] - x_right
                r_r = np.sqrt(raw**2 + dz**2) if raw > 0 else np.nan

            t_out.append(t)
            r_left_out.append(r_l)
            r_right_out.append(r_r)
        except Exception:
            n_fail += 1

    t_arr  = np.array(t_out)
    rl_arr = np.array(r_left_out)
    rr_arr = np.array(r_right_out)
    sh_tracks[b['run_id']] = dict(lambda_bar=lb, t=t_arr,
                                   r_left=rl_arr, r_right=rr_arr,
                                   dx=dx_run)
    valid_l = np.sum(np.isfinite(rl_arr))
    valid_r = np.sum(np.isfinite(rr_arr))
    print(f'[{b["run_id"]}]  dz={dz:.2f}  dx={dx_run:.4f}  '
          f'{len(t_out)} loaded, {n_fail} skipped, '
          f'{valid_l}/{valid_r} (left/right) with wall'
          + (f'  t=[{t_arr[0]:.2f},{t_arr[-1]:.2f}]' if len(t_arr) else ''))

## 3. Plot

In [ ]:
colors  = {0.84: '#0072B2', 0.069: '#D55E00'}
markers = ['o', 's', '^', 'D', 'v']

fig, axes = plt.subplots(1, len(LAMBDA_VALS),
                          figsize=(5.5 * len(LAMBDA_VALS), 4), squeeze=False)

for col, lb in enumerate(LAMBDA_VALS):
    ax     = axes[0, col]
    color  = colors[lb]
    info   = lambda_info[lb]
    rout   = info['rout_0']
    t_coll = info['t_coll']

    # --- 3D tracks (both outer walls, all runs for this lambda) ---
    runs_this_lb = [(b['run_id'], i)
                    for i, b in enumerate(SH_RUNS) if b['lambda_bar'] == lb]
    first = True
    for mk_idx, (run_id, _) in enumerate(runs_this_lb):
        tr   = sh_tracks.get(run_id)
        if tr is None:
            continue
        mk   = markers[mk_idx % len(markers)]
        t    = tr['t']
        yerr = tr['dx'] / 2.   # ±half grid cell — wall-crossing precision

        mask_r = np.isfinite(tr['r_right'])
        if mask_r.any():
            ax.errorbar(t[mask_r], tr['r_right'][mask_r], yerr=yerr,
                        color=color, marker=mk, markersize=3, linestyle='none',
                        capsize=1.5, elinewidth=0.6,
                        label=f'3+1D {run_id} right' if first else f'{run_id} right',
                        zorder=3)

        mask_l = np.isfinite(tr['r_left'])
        if mask_l.any():
            ax.errorbar(t[mask_l], tr['r_left'][mask_l], yerr=yerr,
                        color=color, marker=mk, markersize=3, mfc='none',
                        linestyle='none', capsize=1.5, elinewidth=0.6,
                        label=f'3+1D {run_id} left' if first else f'{run_id} left',
                        zorder=3)
        first = False

    # --- BM track with ±dz/2 band ---
    bm = bm_tracks.get(lb)
    if bm is not None:
        mask  = np.isfinite(bm['r_wall'])
        t_bm  = bm['t'][mask]
        r_bm  = bm['r_wall'][mask]
        half  = bm['dz'] / 2.
        ax.plot(t_bm, r_bm, color=color, lw=1.5, ls='--', label='1+1D', zorder=3)
        ax.fill_between(t_bm, r_bm - half, r_bm + half,
                        color=color, alpha=0.2, linewidth=0, zorder=2)

    # --- Free-expansion reference ---
    t_ref = np.linspace(0, t_coll * 1.6, 400)
    r_ref = np.sqrt(rout**2 + t_ref**2)
    ax.plot(t_ref, r_ref, 'k:', lw=1,
            label=r'free: $\sqrt{r_\mathrm{out}^2+t^2}$', zorder=2)

    ax.axvline(t_coll, color='grey', lw=0.8, ls=':',
               label=f'$t_\\mathrm{{coll}}={t_coll:.2f}$')
    ax.set_xlabel('Minkowski time $t$ (from nucleation)')
    ax.set_ylabel('Outer wall radius from bubble centre')
    ax.set_title(rf'$\bar\lambda = {lb}$,  filled=right / open=left outer wall',
                 fontsize=8)
    ax.legend(frameon=False, fontsize=6.5, ncol=2)

plt.tight_layout()
plt.savefig(REPO_ROOT / 'bubble_wall_tracking.pdf', bbox_inches='tight')
plt.show()

## 4. Deviation from free expansion

In [ ]:
fig, axes = plt.subplots(1, len(LAMBDA_VALS),
                          figsize=(5.5 * len(LAMBDA_VALS), 4), squeeze=False)

for col, lb in enumerate(LAMBDA_VALS):
    ax     = axes[0, col]
    color  = colors[lb]
    info   = lambda_info[lb]
    rout   = info['rout_0']
    t_coll = info['t_coll']

    runs_this_lb = [(b['run_id'], i)
                    for i, b in enumerate(SH_RUNS) if b['lambda_bar'] == lb]
    for mk_idx, (run_id, _) in enumerate(runs_this_lb):
        tr = sh_tracks.get(run_id)
        if tr is None:
            continue
        mk     = markers[mk_idx % len(markers)]
        t      = tr['t']
        r_free = np.sqrt(rout**2 + t**2)
        yerr   = tr['dx'] / 2.   # ±half grid cell

        mask_r = np.isfinite(tr['r_right'])
        if mask_r.any():
            ax.errorbar(t[mask_r], np.abs(tr['r_right'][mask_r] - r_free[mask_r]),
                        yerr=yerr, color=color, marker=mk, markersize=3,
                        linestyle='none', capsize=1.5, elinewidth=0.6,
                        label=f'{run_id} right', zorder=3)

        mask_l = np.isfinite(tr['r_left'])
        if mask_l.any():
            ax.errorbar(t[mask_l], np.abs(tr['r_left'][mask_l] - r_free[mask_l]),
                        yerr=yerr, color=color, marker=mk, markersize=3,
                        mfc='none', linestyle='none', capsize=1.5, elinewidth=0.6,
                        label=f'{run_id} left', zorder=3)

    # BM deviation with ±dz/2 band
    bm = bm_tracks.get(lb)
    if bm is not None:
        mask      = np.isfinite(bm['r_wall'])
        t_bm      = bm['t'][mask]
        r_free_bm = np.sqrt(rout**2 + t_bm**2)
        delta_bm  = np.abs(bm['r_wall'][mask] - r_free_bm)
        half      = bm['dz'] / 2.
        ax.plot(t_bm, delta_bm, color=color, lw=1.5, ls='--', label='1+1D', zorder=3)
        ax.fill_between(t_bm, np.maximum(delta_bm - half, 0), delta_bm + half,
                        color=color, alpha=0.2, linewidth=0, zorder=2)

    ax.axvline(t_coll, color='grey', lw=0.8, ls=':')
    ax.set_yscale('log')
    ax.set_xlabel('Minkowski time $t$')
    ax.set_ylabel(r'$|r_\mathrm{wall} - \sqrt{r_\mathrm{out}^2 + t^2}|$')
    ax.set_title(rf'$\bar\lambda = {lb}$ — deviation from free expansion',
                 fontsize=8)
    ax.legend(frameon=False, fontsize=6.5, ncol=2)

plt.tight_layout()
plt.savefig(REPO_ROOT / 'bubble_wall_delta.pdf', bbox_inches='tight')
plt.show()

## 5. Time offset between 3D and 1D wall tracks (lb=0.84)

In [ ]:
from scipy.interpolate import interp1d

lb = 0.84
bm  = bm_tracks[lb]
info = lambda_info[lb]

# Build monotone BM r(t) and its inverse t(r) over the valid range.
mask_bm = np.isfinite(bm['r_wall']) & (bm['r_wall'] > 0)
t_bm_v  = bm['t'][mask_bm]
r_bm_v  = bm['r_wall'][mask_bm]

# r_wall(t) is monotonically increasing after nucleation; drop any non-monotone
# leading points (very early snapshots where the wall isn't yet resolved).
mono = np.concatenate(([True], np.diff(r_bm_v) > 0))
t_bm_v, r_bm_v = t_bm_v[mono], r_bm_v[mono]

t_of_r_bm = interp1d(r_bm_v, t_bm_v, kind='linear', bounds_error=False, fill_value=np.nan)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax_dt, ax_r = axes

runs_lb = [b['run_id'] for b in SH_RUNS if b['lambda_bar'] == lb]
print(f'lb={lb}  BM r range: [{r_bm_v.min():.3f}, {r_bm_v.max():.3f}]')
print(f'{"run":<22}  {"side":<5}  {"N pts":>5}  {"mean Δt":>9}  {"std Δt":>9}  {"median Δt":>10}')

for mk_idx, run_id in enumerate(runs_lb):
    tr = sh_tracks.get(run_id)
    if tr is None:
        continue
    mk    = markers[mk_idx % len(markers)]
    color = colors[lb]
    t_sh  = tr['t']

    for side, r_sh in [('right', tr['r_right']), ('left', tr['r_left'])]:
        mfc   = color if side == 'right' else 'none'
        mask  = np.isfinite(r_sh)
        if not mask.any():
            continue

        t_bm_interp = t_of_r_bm(r_sh[mask])
        valid = np.isfinite(t_bm_interp)
        if not valid.any():
            continue

        dt    = t_sh[mask][valid] - t_bm_interp[valid]
        r_pts = r_sh[mask][valid]

        ax_dt.errorbar(t_sh[mask][valid], dt, yerr=tr['dx']/2.,
                       color=color, marker=mk, markersize=3, mfc=mfc,
                       linestyle='none', capsize=1.5, elinewidth=0.6,
                       label=f'{run_id} {side}', zorder=3)
        ax_r.errorbar(r_pts, dt, yerr=tr['dx']/2.,
                      color=color, marker=mk, markersize=3, mfc=mfc,
                      linestyle='none', capsize=1.5, elinewidth=0.6,
                      label=f'{run_id} {side}', zorder=3)

        print(f'{run_id:<22}  {side:<5}  {valid.sum():>5}  '
              f'{dt.mean():>+9.4f}  {dt.std():>9.4f}  {np.median(dt):>+10.4f}')

ax_dt.axhline(0, color='grey', lw=0.8, ls=':')
ax_dt.set_xlabel('Minkowski time $t$ (3D)')
ax_dt.set_ylabel(r'$\Delta t = t_\mathrm{3D} - t_\mathrm{BM}(r)$')
ax_dt.set_title(rf'Time offset vs $t$   ($\bar\lambda={lb}$)', fontsize=9)
ax_dt.legend(frameon=False, fontsize=6.5, ncol=2)

ax_r.axhline(0, color='grey', lw=0.8, ls=':')
ax_r.set_xlabel('Outer wall radius $r$')
ax_r.set_ylabel(r'$\Delta t = t_\mathrm{3D} - t_\mathrm{BM}(r)$')
ax_r.set_title(rf'Time offset vs $r$   ($\bar\lambda={lb}$)', fontsize=9)
ax_r.legend(frameon=False, fontsize=6.5, ncol=2)

plt.tight_layout()
plt.savefig(REPO_ROOT / 'bubble_wall_time_offset.pdf', bbox_inches='tight')
plt.show()